# GS STARR Track 1 SEMDB — Steps 01–05 + Donor Extent Analysis

- Step 01: raster extraction + clip donor I/O.
- Step 02: low-RAM matching + hard calipers + SMD.
- Step 03: twin test / parallel trends.
- Step 04: locked Reference Area.
- Step 05: 90% Confidence Interval + `UNCBSL`.
- **Step EXT**: Donor Extent Analysis — confronto `5km | 10km | 20km | 30km | FULL`.

**Istruzioni:** compilare solo la cella **§1 — Parametri**, poi eseguire le celle in ordine.


---
## §0. Setup ambiente


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['figure.figsize'] = (13, 5)

from google.colab import drive
drive.mount('/content/drive')

!pip install rasterio pyproj geopandas shapely scikit-learn scipy pyarrow -q
print('Environment OK')


---
## §1. Parametri

**Modificare solo questa cella.** Tutti i path e i parametri vengono
passati come argomenti espliciti alle funzioni — nessun path hardcoded nei moduli.


In [ ]:
from pathlib import Path
import json, importlib.util, warnings, gc
warnings.filterwarnings('ignore')

# ── Cartelle principali ──────────────────────────────────────────────
SCRIPTS_DIR = Path('/content/drive/MyDrive/Starr_Scripts_V4')
RASTER_DIR  = Path('/content/drive/MyDrive/STARR_Idiofa_New_V2')

# ── Identificatori progetto ──────────────────────────────────────────
# RUN_ID_BASE: solo il nome base, senza path e senza suffisso _extNkm.
# Esempio: 'Idiofa_Lobi_2018_buf50km_excl5km_WRB2_v07_raster'
# Lasciare '' per trovare tutti i TIF covariates_*.tif nella cartella.
RUN_ID_BASE  = 'Idiofa_Lobi_2018_buf50km_excl5km_WRB2_v07_raster'
PROJECT_NAME = 'Idiofa_Lobi'

# ── Shapefile filtro spaziale (impostare None per disattivare) ────────
FNF_SHAPEFILE      = '/content/drive/MyDrive/New_Eligibility/02_Shapefile/Eligibility/FNF18_fullBuffer.shp'
ELIGIBLE_SHAPEFILE = '/content/drive/MyDrive/New_Eligibility/02_Shapefile/Eligibility/Eligible_FNF_fullBuffer.shp'

# ── Donor Extent ─────────────────────────────────────────────────────
# Estensione del pool donor attorno al bordo PA.
#   'full' -> nessun clip   |   5 / 10 / 20 / 30 -> buffer in km
# Impostare dopo aver letto la raccomandazione in §2b.
DONOR_EXTENT_KM = 10

# Estensioni da confrontare nell'analisi §2b (Donor Extent Analysis).
COMPARE_EXTENTS = ['full', 5, 10, 20, 30]

# ── Step 02 — parametri matching ─────────────────────────────────────
K_NEIGHBOURS         = 1
KNN_QUERY_CANDIDATES = 150
N_DONOR_SAMPLE       = 300_000
KNN_N_JOBS           = -1

# ── Step 05 ──────────────────────────────────────────────────────────
RUN_STEP_05 = False   # True solo quando disponibili i dati carbon-stock-change

# ── Output (calcolato automaticamente) ───────────────────────────────
_ext_sfx    = '' if DONOR_EXTENT_KM == 'full' else f'_ext{DONOR_EXTENT_KM}km'
_run_label  = RUN_ID_BASE if RUN_ID_BASE else 'run'
OUTPUT_ROOT    = RASTER_DIR / 'STARR_outputs' / (_run_label + _ext_sfx)
COMPARISON_DIR = RASTER_DIR / 'STARR_outputs' / 'comparison'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

print('SCRIPTS_DIR   :', SCRIPTS_DIR)
print('RASTER_DIR    :', RASTER_DIR)
print('RUN_ID_BASE   :', RUN_ID_BASE or '(wildcard)')
print('FNF_SHAPEFILE :', FNF_SHAPEFILE or 'OFF')
print('ELIGIBLE_SHP  :', ELIGIBLE_SHAPEFILE or 'OFF')
print('DONOR_EXTENT  :', DONOR_EXTENT_KM)
print('OUTPUT_ROOT   :', OUTPUT_ROOT)


---
## §2. Caricamento moduli


In [ ]:
def load_module(module_name, filename):
    path = SCRIPTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Script non trovato: {path}')
    spec = importlib.util.spec_from_file_location(module_name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

s01  = load_module('s01',  '01_STARR_raster_extract.py')
s02  = load_module('s02',  '02_STARR_matching_data_weights.py')
s03  = load_module('s03',  '03_STARR_twin_test_selection.py')
s04  = load_module('s04',  '04_STARR_reference_area_lock.py')
s00b = load_module('s00b', '00b_STARR_compare_extents.py')

# Step 05 — opzionale (cerca entrambi i nomi possibili)
s05 = None
for _fn05 in ['05_STARR_baseline_uncertainty.py',
               '05_STARR_baseline_confidence_interval_UNCBSL.py']:
    if (SCRIPTS_DIR / _fn05).exists():
        s05 = load_module('s05', _fn05)
        break
if s05 is None and RUN_STEP_05:
    print('[WARN] Script Step 05 non trovato — RUN_STEP_05 ignorato.')

# ── Override parametri Step 02 da §1 ─────────────────────────────────
s02.K_NEIGHBOURS         = K_NEIGHBOURS
s02.KNN_QUERY_CANDIDATES = KNN_QUERY_CANDIDATES
s02.N_DONOR_SAMPLE       = N_DONOR_SAMPLE
s02.KNN_N_JOBS           = KNN_N_JOBS

loaded = 's01 | s02 | s03 | s04 | s00b'
if s05: loaded += ' | s05'
print('Loaded:', loaded)


---
## §2b. Donor Extent Analysis *(opzionale)*

Esegue Step 01–04 per ogni extent in `COMPARE_EXTENTS` e determina
l'**extent minimo sufficiente** su 4 criteri:

| Criterio | Soglia |
|---|---|
| `n_donor / n_project` | ≥ 3× |
| `match_coverage_pct` | ≥ 90% |
| `twin_pass_pct` | ≥ 30% |
| `smd_max` | ≤ 0.10 |

> ⚠️ Può richiedere ore. Usare **[EXT-LOAD]** se il confronto è già stato eseguito.


In [ ]:
# ── [EXT-RUN] Esegue il confronto completo ───────────────────────────
# Commentare questa cella se si vogliono caricare risultati esistenti.

comparison_result = s00b.run_comparison(
    extents            = COMPARE_EXTENTS,
    base_dirs          = [RASTER_DIR],
    output_dir         = COMPARISON_DIR,
    run_id_base        = RUN_ID_BASE,
    fnf_shapefile      = FNF_SHAPEFILE,
    eligible_shapefile = ELIGIBLE_SHAPEFILE,
    run_step_05        = False,
)
_metrics_list   = comparison_result['metrics']
_min_suff       = comparison_result['minimum_sufficient']
_recommendation = comparison_result['recommendation']
print(f'Completato: {len(_metrics_list)} extent testati')


In [ ]:
# ── [EXT-LOAD] Carica risultati già prodotti ──────────────────────────
# Alternativa a [EXT-RUN].

_summary_path = COMPARISON_DIR / 'comparison_summary.json'
if not _summary_path.exists():
    print(f'File non trovato: {_summary_path}')
    print('Eseguire prima la cella [EXT-RUN].')
else:
    with open(_summary_path) as _f:
        _comp_data = json.load(_f)
    _metrics_list   = _comp_data['metrics']
    _min_suff       = _comp_data.get('minimum_sufficient')
    _recommendation = _comp_data.get('recommendation', '')
    print(f'Caricato: {len(_metrics_list)} extent  |  {_comp_data["timestamp_utc"]}')


In [ ]:
# ── [EXT-TABLE] Tabella riepilogativa ────────────────────────────────
import pandas as pd, numpy as np

_rows = []
for _m in _metrics_list:
    if 'error' in _m:
        _rows.append({'Extent': str(_m['extent_km']), 'ERRORE': _m['error']})
        continue
    _c = _m['criteria']
    _rows.append({
        'Extent':     'FULL' if _m['extent_km'] == 'full' else f"{_m['extent_km']}km",
        'n_donor':    f"{_m['n_donor']:,}",
        'ratio':      f"{_m['ratio_donor_project']:.1f}x",
        'match%':     f"{_m['match_coverage_pct']:.1f}%",
        'twin%':      f"{_m['twin_pass_pct']:.1f}%",
        'SMD_max':    f"{_m['smd_max']:.3f}" if _m.get('smd_max') is not None else 'N/A',
        'ref_ha':     _m.get('reference_area_ha', '-'),
        'ratio>=3x':  'OK' if _c.get('ratio_3x')        else '--',
        'match>=90%': 'OK' if _c.get('match_cov_90pct') else '--',
        'twin>=30%':  'OK' if _c.get('twin_pass_30pct') else '--',
        'SMD<=0.10':  'OK' if _c.get('smd_le_010')      else '--',
        'ALL OK':     'SI' if _c.get('ALL_SUFFICIENT')   else 'no',
    })

def _color(val):
    if str(val) in ('OK', 'SI'):  return 'background-color:#c8e6c9;color:#1b5e20'
    if str(val) in ('--', 'no'): return 'background-color:#ffcdd2;color:#b71c1c'
    return ''

_bool_cols = ['ratio>=3x', 'match>=90%', 'twin>=30%', 'SMD<=0.10', 'ALL OK']
display(pd.DataFrame(_rows).style
        .applymap(_color, subset=_bool_cols)
        .set_caption('GS STARR — Donor Extent Comparison'))


In [ ]:
# ── [EXT-PLOT] Plot comparativo 4-panel + heatmap ────────────────────
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

_valid  = [_m for _m in _metrics_list if 'error' not in _m]
_labels = ['FULL' if _m['extent_km'] == 'full' else f"{_m['extent_km']}km" for _m in _valid]
_x      = np.arange(len(_labels))
_bkw    = dict(width=0.55, edgecolor='white', zorder=3)
_gc     = lambda ok: '#2e7d32' if ok else '#c62828'

_fig = plt.figure(figsize=(max(16, len(_labels)*3.2), 14), facecolor='white')
_gs  = GridSpec(2, 4, figure=_fig, hspace=0.55, wspace=0.38, height_ratios=[2.5, 1])

for _i, (_key, _title, _thresh, _crit, _better) in enumerate([
    ('ratio_donor_project', 'A — Donor/Project ratio (>=3x)', 3.0,  'ratio_3x',        'high'),
    ('match_coverage_pct',  'B — Match coverage % (>=90)',    90.0, 'match_cov_90pct', 'high'),
    ('twin_pass_pct',       'C — Twin pass % (>=30)',         30.0, 'twin_pass_30pct', 'high'),
    ('smd_max',             'D — SMD max (<=0.10)',           0.10, 'smd_le_010',      'low'),
]):
    _ax   = _fig.add_subplot(_gs[0, _i])
    _vals = [float(_m.get(_key) or 0) for _m in _valid]
    _cols = [_gc(_m['criteria'].get(_crit, False)) for _m in _valid]
    _ax.bar(_x, _vals, color=_cols, alpha=0.82, **_bkw)
    _ax.axhline(_thresh, color='#f5a623', ls='--', lw=1.6, label=f'Soglia {_thresh}')
    _ax.set_xticks(_x); _ax.set_xticklabels(_labels, fontsize=8)
    _ax.set_title(_title, fontsize=9, fontweight='bold')
    _ax.legend(fontsize=7); _ax.grid(axis='y', alpha=0.3, zorder=0)
    for _xi, _vi in zip(_x, _vals):
        _ax.text(_xi, _vi, f'{_vi:.2f}', ha='center', va='bottom', fontsize=7)

_ax2   = _fig.add_subplot(_gs[1, :])
_ckeys = ['ratio_3x', 'match_cov_90pct', 'twin_pass_30pct', 'smd_le_010', 'ALL_SUFFICIENT']
_clbls = ['Ratio>=3x', 'Match>=90%', 'Twin>=30%', 'SMD<=0.10', 'TUTTO OK']
_heat  = np.array([[1. if _m['criteria'].get(_k) else 0. for _m in _valid] for _k in _ckeys])
_ax2.imshow(_heat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
_ax2.set_xticks(range(len(_valid)));  _ax2.set_xticklabels(_labels, fontsize=9)
_ax2.set_yticks(range(len(_ckeys))); _ax2.set_yticklabels(_clbls, fontsize=9)
for _i in range(len(_ckeys)):
    for _j in range(len(_valid)):
        _ax2.text(_j, _i, 'OK' if _heat[_i, _j] else '--',
                  ha='center', va='center', fontsize=10,
                  color='white' if _heat[_i, _j] == 0 else '#1b5e20', fontweight='bold')
if _min_suff:
    _ml = 'FULL' if _min_suff['extent_km'] == 'full' else f"{_min_suff['extent_km']}km"
    if _ml in _labels:
        _j0 = _labels.index(_ml)
        for _ii in range(len(_ckeys)):
            _ax2.add_patch(plt.Rectangle((_j0-.5, _ii-.5), 1, 1,
                           fill=False, edgecolor='#0d47a1', lw=3))
_ax2.set_title('Heatmap criteri (bordo blu = extent minimo sufficiente)', fontsize=9, fontweight='bold')
_fig.suptitle('GS STARR — Donor Extent Comparison', fontsize=11, fontweight='bold')
plt.tight_layout()
_fig.savefig(COMPARISON_DIR / 'comparison_plots_notebook.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# ── [EXT-TREND] Andamento metriche vs distanza ───────────────────────
_num  = [_m for _m in _valid if _m['extent_km'] != 'full']
_full = [_m for _m in _valid if _m['extent_km'] == 'full']

if _num:
    _xs = [float(_m['extent_km']) for _m in _num]
    _fig2, _axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='white')
    for _key, _title, _thresh, _ax in [
        ('match_coverage_pct', 'Match coverage %', 90.0, _axes[0]),
        ('twin_pass_pct',      'Twin pass %',       30.0, _axes[1]),
        ('smd_max',            'SMD max',            0.10, _axes[2]),
    ]:
        _ys = [float(_m.get(_key) or 0) for _m in _num]
        _ax.plot(_xs, _ys, 'o-', lw=2, ms=8, color='#1565c0', zorder=3)
        _ax.axhline(_thresh, color='#f5a623', ls='--', lw=1.5, label=f'Soglia {_thresh}')
        if _full:
            _fy = float(_full[0].get(_key) or 0)
            _ax.axhline(_fy, color='#7b1fa2', ls=':', lw=1.5, label=f'FULL={_fy:.2f}')
        _ax.set_xlabel('Buffer donor (km)', fontsize=9)
        _ax.set_title(_title, fontsize=9, fontweight='bold')
        _ax.legend(fontsize=7); _ax.grid(alpha=0.3)
    _fig2.suptitle('Andamento metriche vs extent donor', fontsize=10, fontweight='bold')
    plt.tight_layout()
    _fig2.savefig(COMPARISON_DIR / 'metrics_vs_extent.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()


In [ ]:
# ── [EXT-DECISION] Raccomandazione ───────────────────────────────────
print('=' * 65)
print('RACCOMANDAZIONE DONOR EXTENT')
print('=' * 65)
print(_recommendation)
print()
if _min_suff:
    _rec_ext = _min_suff['extent_km']
    print(f'-> Impostare in §1:  DONOR_EXTENT_KM = {repr(_rec_ext)}')
    print(f"   n_donor        : {_min_suff['n_donor']:,}")
    print(f"   ratio          : {_min_suff['ratio_donor_project']:.1f}x")
    print(f"   match_coverage : {_min_suff['match_coverage_pct']:.1f}%")
    print(f"   twin_pass      : {_min_suff['twin_pass_pct']:.1f}%")
    print(f"   smd_max        : {_min_suff['smd_max']:.3f}")
    print(f"   reference_area : {_min_suff.get('reference_area_ha', 'N/A')} ha")
else:
    print('Nessun extent soddisfa tutti i criteri.')
    print('Aggiungere estensioni maggiori a COMPARE_EXTENTS e rieseguire [EXT-RUN].')
print('=' * 65)


---
## §3. Step 01 — Raster extraction

`DONOR_EXTENT_KM`, `RUN_ID_BASE`, `FNF_SHAPEFILE` e `ELIGIBLE_SHAPEFILE`
vengono passati come argomenti — nessun path hardcoded nel modulo.


In [ ]:
proj_df, donor_df, meta, out01 = s01.run_extraction(
    base_dirs          = [RASTER_DIR],
    donor_extent_km    = DONOR_EXTENT_KM,
    run_id_base        = RUN_ID_BASE,
    fnf_shapefile      = FNF_SHAPEFILE,
    eligible_shapefile = ELIGIBLE_SHAPEFILE,
    verbose            = True,
)

# OUTPUT_ROOT si aggiorna con il run_id effettivo prodotto da Step 01
OUTPUT_ROOT = out01.parent

print('Step 01 output   :', out01)
print('Project pixels   :', f'{len(proj_df):,}')
print('Donor pixels     :', f'{len(donor_df):,}')
print('Ratio donor/proj :', f'{len(donor_df)/max(1,len(proj_df)):.1f}x')
print('T0 year          :', meta.get('t0_year'))
print('NDVI years       :', meta.get('year_list'))
print('Covariates       :', meta.get('continuous_covariates'))
print('Donor extent     :', meta.get('donor_extent_label', 'FULL'))


---
## §4. Step 02 — Matching low-RAM + SMD

`donor_df=None` — il donor viene riletto in streaming dal parquet di Step 01,
senza tenerlo in RAM. I parametri KNN sono già stati impostati in §2.


In [ ]:
# donor_df non serve in RAM: Step 02 rilegge il parquet in streaming
del donor_df; gc.collect()

matched_df, weights_dict, imp_df, smd_df, figs02, out02 = s02.run_matching_step(
    base_dirs  = [out01],
    output_dir = OUTPUT_ROOT / '02_matching',
    proj_df    = proj_df,
    donor_df   = None,
    meta       = meta,
    verbose    = True,
)
print('Step 02 output:', out02)
print('Matched rows  :', f'{len(matched_df):,}')
print('SMD max       :', round(smd_df['SMD'].max(), 4) if not smd_df.empty else 'N/A')
display(smd_df)


---
## §5. Step 03 — Twin test / Parallel trends


In [ ]:
all_pairs, twin_pixels, twin_report, figs03, out03 = s03.run_twin_test(
    base_dirs  = [out02],
    output_dir = OUTPUT_ROOT / '03_twin_test',
    matched_df = matched_df,
    proj_df    = proj_df,
    meta       = meta,
    verbose    = True,
)
print('Step 03 output     :', out03)
print('Twin-tested pixels :', f'{len(twin_pixels):,}')
print('Aggregate passed   :', twin_report.get('aggregate_twin_passed'))
print(json.dumps(twin_report, indent=2)[:4000])


---
## §6. Step 04 — Locked Reference Area


In [ ]:
del proj_df; gc.collect()

bounds_gdf, mon_gdf, fig04, out04, manifest = s04.run_reference_area_lock(
    base_dirs   = [out03],
    output_dir  = OUTPUT_ROOT / '04_reference_area',
    passed_df   = twin_pixels,
    meta        = meta,
    twin_report = twin_report,
    verbose     = True,
)
print('Step 04 output    :', out04)
print('RA features       :', len(bounds_gdf))
print('Monitoring points :', len(mon_gdf))
_ra = manifest.get('reference_area_definition', {})
print('Reference area ha :', _ra.get('total_ha'))
print('Lock status       :', manifest.get('lock_status'))
print(json.dumps(manifest, indent=2)[:4000])


---
## §7. Step 05 — Baseline 90% CI / UNCBSL *(opzionale)*

Eseguire solo quando disponibili i dati carbon-stock-change dei pixel controllo locked.
File accettato: `control_carbon_stock_change.parquet` o `.csv` in `OUTPUT_ROOT`.
Impostare `RUN_STEP_05 = True` in §1 per abilitare.


In [ ]:
if not RUN_STEP_05:
    print('Step 05 saltato (RUN_STEP_05=False in §1).')
elif s05 is None:
    print('[ERR] Script Step 05 non trovato in SCRIPTS_DIR.')
else:
    import pandas as pd
    control_pixels, ci_summary, ci_report, fig05, out05 = s05.calculate_uncbsl(
        base_dirs         = [OUTPUT_ROOT],
        output_dir        = OUTPUT_ROOT / '05_baseline_CI_UNCBSL',
        twin_df           = twin_pixels,
        control_change_df = None,
        manifest          = manifest,
        verbose           = True,
    )
    print('Step 05 output:', out05)
    display(control_pixels.head())
    display(pd.DataFrame([ci_summary]))


---
## §8. Output checklist


In [ ]:
print(f'Output root: {OUTPUT_ROOT}')
print()
for _dname, _label in [
    ('01_extract',           'Step 01 — Raster extraction'),
    ('02_matching',          'Step 02 — Matching + SMD'),
    ('03_twin_test',         'Step 03 — Twin test'),
    ('04_reference_area',    'Step 04 — Locked Reference Area'),
    ('05_baseline_CI_UNCBSL','Step 05 — Baseline CI / UNCBSL'),
]:
    _p = OUTPUT_ROOT / _dname
    if _p.exists():
        _files = sorted(_p.iterdir())
        print(f'[OK] {_label} ({len(_files)} file)')
        for _f in _files: print(f'       {_f.name}')
    else:
        print(f'[-] {_label}  — non ancora prodotta')
    print()

print(f'Comparison dir: {COMPARISON_DIR}')
if COMPARISON_DIR.exists():
    for _f in sorted(COMPARISON_DIR.iterdir()):
        print(f'    {_f.name}')
else:
    print('    (non ancora prodotta — eseguire §2b)')
